# 05 — Ramificación y Poda *(Branch & Bound)*
**Algoritmos y Estructuras de Datos · Universidad de Talca**

---
> *"Branch and Bound es Backtracking con memoria: recuerda cuánto vale la mejor solución encontrada y descarta cualquier rama que no pueda superarla."*

| Campo | Detalle |
|---|---|
| **Tópico** | S03 — Diseño de Algoritmos |
| **Notebook** | 05 de 06 |
| **Duración estimada** | 90 minutos |
| **Prerequisito** | NB04 — Backtracking |
| **Paradigma central** | Backtracking + función de cota (upper bound) |

---
### Hilo conductor: la mochila de los 4 paradigmas
Los mismos 5 objetos del PDF han sido resueltos en:
- **NB02** — Greedy (resultado subóptimo para 0/1)
- **NB03** — Programación Dinámica (óptimo, tabla 2D)
- **NB04** — Backtracking (óptimo, árbol completo)
- **NB05** ← este notebook — Branch & Bound (óptimo, **árbol podado**)

In [ ]:
# ── Verificación de dependencias ────────────────────────────────────────────
import sys

_reqs = {'numpy': 'numpy', 'matplotlib': 'matplotlib', 'ipywidgets': 'ipywidgets'}
_missing = []
for pkg, mod in _reqs.items():
    try:
        __import__(mod)
        print(f'✓ {pkg}')
    except ImportError:
        _missing.append(pkg)
        print(f'✗ {pkg} — falta')

if _missing:
    pkgs = ' '.join(_missing)
    print(f'\nInstalar: !pip install {pkgs}')
else:
    ver = sys.version.split()[0]
    print(f'\nPython {ver} — listo.')

## 1. Objetivos de aprendizaje

Al terminar este notebook el estudiante será capaz de:

1. **Explicar** la diferencia entre poda por *inviabilidad* (Backtracking) y poda por *suboptimalidad* (B&B).
2. **Diseñar** una función de cota superior para el problema de la mochila 0/1.
3. **Implementar** Branch & Bound para la mochila usando la relajación fraccionaria como cota.
4. **Comparar visualmente** los árboles de Backtracking puro y B&B sobre los mismos datos.
5. **Resolver** el 8-puzzle usando B&B con heurística de distancia Manhattan.

## 2. Motivación — ¿Por qué Backtracking no es suficiente?

Backtracking explora el espacio de soluciones podando ramas **inviables** (las que violan restricciones).  
Pero no descarta ramas que son *viables* aunque **nunca podrán superar** la mejor solución ya encontrada.

**Ejemplo mental:** Buscas el mejor camino de Santiago a Temuco.  
Ya encontraste un camino de 600 km. Backtracking no descartará un camino parcial de 580 km... aunque ya sabes que ese camino debe pasar por una zona remota que **necesariamente** agrega 100 km más. Branch & Bound sí lo descarta: sabe que la mejor continuación posible da 680 km > 600 km → poda.

### Las dos podas de B&B

| Tipo de poda | Condición | Paradigma |
|---|---|---|
| **Inviabilidad** | La solución parcial viola una restricción (ej: excede capacidad) | Backtracking |
| **Suboptimalidad** | La *mejor extensión posible* de esta rama ≤ mejor solución actual | Branch & Bound |

B&B incluye ambas podas. Backtracking solo tiene la primera.

## 3. Teoría — Función de cota y estrategias de búsqueda

### 3.1 Función de cota superior (*upper bound*)

Para podar por suboptimalidad necesitamos responder: *"¿cuánto podría valer la mejor solución que pasa por este nodo?"*

La cota debe ser **admisible**: nunca subestimar el valor real (si subestimara, podría descartar la solución óptima).

Para la mochila 0/1, la cota más popular es la **relajación fraccionaria**:  
→ ¿Cuánto obtendríamos si pudiéramos tomar *fracciones* de los objetos restantes?  
→ Esto nunca es menor que el óptimo real (la versión 0/1 es más restrictiva).  
→ Se calcula en **O(n)** ordenando objetos por ratio valor/peso.

```
cota_superior(nivel, peso_usado, valor_acumulado):
    bound = valor_acumulado
    resto = capacidad - peso_usado
    para cada objeto i desde nivel en orden de ratio:
        si objeto[i].peso ≤ resto:
            bound += objeto[i].valor
            resto -= objeto[i].peso
        sino:
            bound += (resto / objeto[i].peso) × objeto[i].valor
            break
    return bound
```

### 3.2 Estrategias de recorrido del árbol

| Estrategia | Estructura | Ventaja | Desventaja |
|---|---|---|---|
| **DFS** (pila) | Stack LIFO | Poca memoria | Puede explorar ramas malas primero |
| **BFS** (cola) | Queue FIFO | Encuentra solución óptima por nivel | Mucha memoria |
| **Best-first** | Cola de prioridad (heap) | Explora primero los nodos más prometedores | Requiere buena función de cota |

En este notebook usamos **DFS recursivo** (simple, baja memoria) con poda B&B aplicada en cada nodo.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display
import ipywidgets as widgets
from ipywidgets import Output, IntSlider, Checkbox, HBox, VBox
import heapq
from typing import List, Dict, Tuple, Optional

# ── Paleta de colores del curso ─────────────────────────────────────────────
AZUL     = '#2196F3'
NARANJA  = '#FF9800'
VERDE    = '#4CAF50'
ROJO     = '#F44336'
MORADO   = '#9C27B0'
AZ_CLARO = '#90CAF9'
FONDO    = '#FAFAFA'
TEXTO    = '#212121'

# ── Datos del PDF (idénticos a notebooks 02, 03 y 04) ────────────────────────
OBJETOS = [
    {'nombre': 'Objeto 1', 'valor': 20, 'peso': 50},
    {'nombre': 'Objeto 2', 'valor': 24, 'peso': 100},
    {'nombre': 'Objeto 3', 'valor': 55, 'peso': 150},
    {'nombre': 'Objeto 4', 'valor': 40, 'peso': 200},
    {'nombre': 'Objeto 5', 'valor': 70, 'peso': 250},
]
CAPACIDAD_W = 300

print('✓ Constantes cargadas')
print(f'  {len(OBJETOS)} objetos | W = {CAPACIDAD_W}')
print(f'  {"Objeto":<10} {"Valor":>6} {"Peso":>6} {"Ratio":>8}')
print('  ' + '-' * 34)
for o in OBJETOS:
    nom = o['nombre']
    val = o['valor']
    pes = o['peso']
    rat = val / pes
    print(f'  {nom:<10} {val:>6} {pes:>6} {rat:>8.4f}')

In [ ]:
# ── Función de cota superior (relajación fraccionaria) ─────────────────────

def ordenar_por_ratio(objetos: List[Dict]) -> List[Dict]:
    '''Ordena objetos por ratio valor/peso descendente.'''
    return sorted(objetos, key=lambda x: x['valor'] / x['peso'], reverse=True)


def cota_superior_fraccionaria(
        nivel: int, peso: int, valor: int,
        objetos_ord: List[Dict], capacidad: int) -> float:
    '''
    Cota superior admisible para la mochila 0/1.

    Calcula el valor máximo posible si pudiéramos tomar fracciones
    de los objetos restantes (relajación fraccionaria).
    Siempre es >= al óptimo real → nunca descarta la solución óptima.

    Complejidad:
        Tiempo: O(n) donde n = objetos desde nivel hasta el final
        Espacio: O(1)
    '''
    if peso > capacidad:
        return 0.0
    bound = float(valor)
    restante = capacidad - peso
    for i in range(nivel, len(objetos_ord)):
        obj = objetos_ord[i]
        if obj['peso'] <= restante:
            restante -= obj['peso']
            bound += obj['valor']
        else:
            bound += (restante / obj['peso']) * obj['valor']
            break
    return bound


# Demostración
objs_ord = ordenar_por_ratio(OBJETOS)
print('Objetos ordenados por ratio (valor/peso):')
for i, o in enumerate(objs_ord):
    nom = o['nombre']
    val = o['valor']
    pes = o['peso']
    rat = val / pes
    print(f'  [{i}] {nom}: ratio={rat:.4f}, val={val}, peso={pes}')

c0 = cota_superior_fraccionaria(0, 0, 0, objs_ord, CAPACIDAD_W)
print(f'\nCota superior desde raíz (sin nada tomado): {c0:.2f}')
print('Interpretación: el óptimo real nunca puede superar este valor.')

In [ ]:
# ── Layout de árbol (algoritmo Reingold-Tilford simplificado) ──────────────

def calcular_posiciones_arbol(nodos: List[Dict]) -> Dict:
    '''Calcula posiciones (x, y) normalizadas a [0,1] para dibujar el árbol.'''
    if not nodos:
        return {}

    ids_validos = {n['id'] for n in nodos}
    hijos: Dict[int, List[int]] = {n['id']: [] for n in nodos}
    raiz_id = None

    for n in nodos:
        p = n['padre']
        if p is None or p not in ids_validos:
            raiz_id = n['id']
        elif p in hijos:
            hijos[p].append(n['id'])

    if raiz_id is None:
        raiz_id = nodos[0]['id']

    posiciones: Dict[int, Tuple[float, float]] = {}
    contador = [0]

    def dfs(nid: int, nivel: int) -> None:
        ch = hijos.get(nid, [])
        if not ch:
            x = float(contador[0])
            contador[0] += 1
        else:
            for h in ch:
                dfs(h, nivel + 1)
            xs = [posiciones[h][0] for h in ch]
            x = sum(xs) / len(xs)
        posiciones[nid] = (x, float(-nivel))

    dfs(raiz_id, 0)

    for n in nodos:
        if n['id'] not in posiciones:
            posiciones[n['id']] = (float(contador[0]), float(-n['nivel']))
            contador[0] += 1

    xs = [p[0] for p in posiciones.values()]
    ys = [p[1] for p in posiciones.values()]
    x_min, x_max = min(xs), max(xs)
    y_min, y_max = min(ys), max(ys)
    dx = max(x_max - x_min, 0.01)
    dy = max(y_max - y_min, 0.01)

    return {
        nid: ((x - x_min) / dx, (y - y_min) / dy)
        for nid, (x, y) in posiciones.items()
    }


# ── Backtracking puro con registro de árbol ────────────────────────────────

def mochila_bt_con_arbol(objetos: List[Dict], capacidad: int):
    '''
    Mochila 0/1 por Backtracking puro.
    Registra el árbol de exploración para visualización.

    Complejidad:
        Tiempo: O(2^n) en el peor caso
        Espacio: O(n) pila de recursión + O(2^n) nodos del árbol
    '''
    objs = ordenar_por_ratio(objetos)
    n = len(objs)
    mejor = [0]
    mejor_items: List[List[str]] = [[]]
    nodos: List[Dict] = []

    def bt(nivel: int, peso: int, valor: int,
           items: List[str], padre_id: Optional[int]) -> None:
        nid = len(nodos)
        nodos.append({'id': nid, 'padre': padre_id, 'nivel': nivel,
                      'valor': valor, 'peso': peso, 'podado': False})
        if valor > mejor[0]:
            mejor[0] = valor
            mejor_items[0] = items[:]
        if nivel >= n:
            return
        obj = objs[nivel]
        # Rama incluir
        if peso + obj['peso'] <= capacidad:
            bt(nivel + 1, peso + obj['peso'], valor + obj['valor'],
               items + [obj['nombre']], nid)
        else:
            pid = len(nodos)
            nodos.append({'id': pid, 'padre': nid, 'nivel': nivel + 1,
                          'valor': valor + obj['valor'],
                          'peso': peso + obj['peso'], 'podado': True})
        # Rama excluir (siempre explorar en BT puro)
        bt(nivel + 1, peso, valor, items, nid)

    bt(0, 0, 0, [], None)
    return mejor[0], mejor_items[0], nodos


# ── Branch & Bound con registro de árbol ──────────────────────────────────

def mochila_bb_con_arbol(objetos: List[Dict], capacidad: int):
    '''
    Mochila 0/1 por Branch & Bound (DFS + cota superior fraccionaria).
    Registra el árbol de exploración para visualización.

    Complejidad:
        Tiempo: O(2^n) peor caso, mucho menor en práctica por la poda
        Espacio: O(n) pila + O(nodos_explorados) árbol
    '''
    objs = ordenar_por_ratio(objetos)
    n = len(objs)
    mejor = [0]
    mejor_items: List[List[str]] = [[]]
    nodos: List[Dict] = []

    def bb(nivel: int, peso: int, valor: int,
           items: List[str], padre_id: Optional[int]) -> None:
        nid = len(nodos)
        cota = cota_superior_fraccionaria(nivel, peso, valor, objs, capacidad)
        nodos.append({'id': nid, 'padre': padre_id, 'nivel': nivel,
                      'valor': valor, 'peso': peso,
                      'cota': cota, 'podado': False})
        if valor > mejor[0]:
            mejor[0] = valor
            mejor_items[0] = items[:]
        if nivel >= n:
            return
        obj = objs[nivel]
        nv = nivel + 1
        # Rama incluir
        np_ = peso + obj['peso']
        nv_ = valor + obj['valor']
        if np_ <= capacidad:
            c = cota_superior_fraccionaria(nv, np_, nv_, objs, capacidad)
            if c > mejor[0]:
                bb(nv, np_, nv_, items + [obj['nombre']], nid)
            else:
                pid = len(nodos)
                nodos.append({'id': pid, 'padre': nid, 'nivel': nv,
                              'valor': nv_, 'peso': np_,
                              'cota': c, 'podado': True})
        else:
            pid = len(nodos)
            nodos.append({'id': pid, 'padre': nid, 'nivel': nv,
                          'valor': nv_, 'peso': np_,
                          'cota': 0.0, 'podado': True})
        # Rama excluir
        c = cota_superior_fraccionaria(nv, peso, valor, objs, capacidad)
        if c > mejor[0]:
            bb(nv, peso, valor, items, nid)
        else:
            pid = len(nodos)
            nodos.append({'id': pid, 'padre': nid, 'nivel': nv,
                          'valor': valor, 'peso': peso,
                          'cota': c, 'podado': True})

    bb(0, 0, 0, [], None)
    return mejor[0], mejor_items[0], nodos


print('✓ Funciones de árbol definidas')

In [ ]:
# ── Ejecutar y comparar ─────────────────────────────────────────────────────
val_bt, items_bt, arbol_bt = mochila_bt_con_arbol(OBJETOS, CAPACIDAD_W)
val_bb, items_bb, arbol_bb = mochila_bb_con_arbol(OBJETOS, CAPACIDAD_W)

nodos_bt   = len(arbol_bt)
podados_bt = sum(1 for n in arbol_bt if n['podado'])
explor_bt  = nodos_bt - podados_bt

nodos_bb   = len(arbol_bb)
podados_bb = sum(1 for n in arbol_bb if n['podado'])
explor_bb  = nodos_bb - podados_bb

reduccion = (1 - nodos_bb / nodos_bt) * 100 if nodos_bt > 0 else 0

print('=' * 62)
print('   COMPARACIÓN: Backtracking Puro  vs  Branch & Bound')
print('=' * 62)
print(f'{"Métrica":<32} {"Backtracking":>12} {"Branch&Bound":>12}')
print('-' * 58)
print(f'{"Valor óptimo":<32} {val_bt:>12} {val_bb:>12}')
print(f'{"Total nodos creados":<32} {nodos_bt:>12} {nodos_bb:>12}')
print(f'{"Nodos podados (inviables)":<32} {podados_bt:>12} {podados_bb:>12}')
print(f'{"Nodos realmente explorados":<32} {explor_bt:>12} {explor_bb:>12}')
print(f'{"Reducción de nodos":<32} {"—":>12} {reduccion:>11.1f}%')
print('=' * 62)
print(f'\nSolución BT : {items_bt}')
print(f'Solución B&B: {items_bb}')

In [ ]:
# ── Visualización comparativa lado a lado ──────────────────────────────────

def dibujar_arbol(ax, nodos: List[Dict], titulo: str) -> None:
    '''Dibuja el árbol de exploración en el eje dado.'''
    if not nodos:
        return
    posiciones = calcular_posiciones_arbol(nodos)
    max_nivel = max(n['nivel'] for n in nodos)

    # Aristas
    for n in nodos:
        p = n['padre']
        if p is not None and p in posiciones and n['id'] in posiciones:
            x0, y0 = posiciones[p]
            x1, y1 = posiciones[n['id']]
            lc = ROJO if n['podado'] else '#BDBDBD'
            lw = 1.0 if n['podado'] else 1.5
            ax.plot([x0, x1], [y0, y1], '-', color=lc, lw=lw, zorder=1, alpha=0.75)

    # Nodos
    for n in nodos:
        if n['id'] not in posiciones:
            continue
        x, y = posiciones[n['id']]
        if n['podado']:
            c, r = ROJO, 0.018
        elif n['padre'] is None:
            c, r = AZUL, 0.024
        elif n['nivel'] == max_nivel:
            c, r = VERDE, 0.018
        else:
            c, r = AZ_CLARO, 0.020
        circ = plt.Circle((x, y), r, color=c, zorder=2)
        ax.add_patch(circ)
        tc = TEXTO if c == AZ_CLARO else 'white'
        ax.text(x, y, str(n['valor']), ha='center', va='center',
                fontsize=5, color=tc, zorder=3, fontweight='bold')

    total = len(nodos)
    podados = sum(1 for n in nodos if n['podado'])
    ax.text(0.01, 0.01, f'Total: {total}  Podados: {podados}',
            transform=ax.transAxes, fontsize=8,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.85))

    legend = [
        mpatches.Patch(color=AZUL,     label='Raíz'),
        mpatches.Patch(color=AZ_CLARO, label='Explorado'),
        mpatches.Patch(color=VERDE,    label='Hoja/solución'),
        mpatches.Patch(color=ROJO,     label='Podado'),
    ]
    ax.legend(handles=legend, loc='lower right', fontsize=7, framealpha=0.9)
    ax.set_xlim(-0.08, 1.08)
    ax.set_ylim(-0.08, 1.1)
    ax.set_title(titulo, fontsize=11, fontweight='bold', color=TEXTO, pad=8)
    ax.axis('off')
    ax.set_facecolor(FONDO)


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 7))
fig.patch.set_facecolor(FONDO)
fig.suptitle('Árbol de exploración: Backtracking vs Branch & Bound\n'
             f'Mochila PDF — {len(OBJETOS)} objetos, W={CAPACIDAD_W}',
             fontsize=13, fontweight='bold', color=TEXTO, y=1.01)

dibujar_arbol(ax1, arbol_bt,
              f'Backtracking puro ({len(arbol_bt)} nodos)')
dibujar_arbol(ax2, arbol_bb,
              f'Branch & Bound ({len(arbol_bb)} nodos) '
              f'— {(1-len(arbol_bb)/len(arbol_bt))*100:.0f}% menos')

plt.tight_layout()
plt.show()

In [ ]:
# ── Animación: exploración B&B paso a paso ─────────────────────────────────

def recopilar_pasos_bb(objetos: List[Dict], capacidad: int):
    '''Ejecuta B&B y devuelve la secuencia de pasos para animación.'''
    objs = ordenar_por_ratio(objetos)
    n = len(objs)
    pasos = []          # (snapshot_nodos, mejor_valor, id_activo)
    nodos_global: List[Dict] = []
    mejor = [0]

    def bb(nivel, peso, valor, items, padre_id):
        nid = len(nodos_global)
        cota = cota_superior_fraccionaria(nivel, peso, valor, objs, capacidad)
        nodos_global.append({'id': nid, 'padre': padre_id, 'nivel': nivel,
                              'valor': valor, 'peso': peso, 'podado': False})
        if valor > mejor[0]:
            mejor[0] = valor
        pasos.append(([dict(x) for x in nodos_global], mejor[0], nid))

        if nivel >= n:
            return
        obj = objs[nivel]
        nv = nivel + 1
        np_ = peso + obj['peso']
        nv_ = valor + obj['valor']

        if np_ <= capacidad:
            c = cota_superior_fraccionaria(nv, np_, nv_, objs, capacidad)
            if c > mejor[0]:
                bb(nv, np_, nv_, items + [obj['nombre']], nid)
            else:
                pid = len(nodos_global)
                nodos_global.append({'id': pid, 'padre': nid, 'nivel': nv,
                                     'valor': nv_, 'peso': np_, 'podado': True})
                pasos.append(([dict(x) for x in nodos_global], mejor[0], pid))
        else:
            pid = len(nodos_global)
            nodos_global.append({'id': pid, 'padre': nid, 'nivel': nv,
                                  'valor': nv_, 'peso': np_, 'podado': True})
            pasos.append(([dict(x) for x in nodos_global], mejor[0], pid))

        c = cota_superior_fraccionaria(nv, peso, valor, objs, capacidad)
        if c > mejor[0]:
            bb(nv, peso, valor, items, nid)
        else:
            pid = len(nodos_global)
            nodos_global.append({'id': pid, 'padre': nid, 'nivel': nv,
                                  'valor': valor, 'peso': peso, 'podado': True})
            pasos.append(([dict(x) for x in nodos_global], mejor[0], pid))

    bb(0, 0, 0, [], None)
    return pasos


pasos_anim = recopilar_pasos_bb(OBJETOS, CAPACIDAD_W)

fig_a, ax_a = plt.subplots(figsize=(10, 6))
fig_a.patch.set_facecolor(FONDO)

def _update_anim(frame):
    ax_a.clear()
    snap, mejor_val, activo = pasos_anim[frame]
    posiciones = calcular_posiciones_arbol(snap)
    max_nivel = max(n['nivel'] for n in snap) if snap else 0

    for n in snap:
        p = n['padre']
        if p is not None and p in posiciones and n['id'] in posiciones:
            x0, y0 = posiciones[p]
            x1, y1 = posiciones[n['id']]
            lc = ROJO if n['podado'] else '#BDBDBD'
            ax_a.plot([x0, x1], [y0, y1], '-', color=lc, lw=1.5, zorder=1, alpha=0.7)

    for n in snap:
        if n['id'] not in posiciones:
            continue
        x, y = posiciones[n['id']]
        if n['id'] == activo:
            c = NARANJA
        elif n['podado']:
            c = ROJO
        elif n['padre'] is None:
            c = AZUL
        elif n['nivel'] == max_nivel:
            c = VERDE
        else:
            c = AZ_CLARO
        circ = plt.Circle((x, y), 0.025, color=c, zorder=2)
        ax_a.add_patch(circ)
        tc = TEXTO if c == AZ_CLARO else 'white'
        ax_a.text(x, y, str(n['valor']), ha='center', va='center',
                  fontsize=7, color=tc, zorder=3, fontweight='bold')

    ley = [
        mpatches.Patch(color=NARANJA,  label='Nodo actual'),
        mpatches.Patch(color=AZ_CLARO, label='Explorado'),
        mpatches.Patch(color=VERDE,    label='Hoja'),
        mpatches.Patch(color=ROJO,     label='Podado B&B'),
        mpatches.Patch(color=AZUL,     label='Raíz'),
    ]
    ax_a.legend(handles=ley, loc='lower right', fontsize=8, framealpha=0.9)
    ax_a.set_xlim(-0.1, 1.1)
    ax_a.set_ylim(-0.12, 1.15)
    ax_a.axis('off')
    ax_a.set_facecolor(FONDO)
    paso = frame + 1
    total = len(pasos_anim)
    n_nodos = len(snap)
    ax_a.set_title(
        f'Branch & Bound — Paso {paso}/{total}  |  '
        f'Nodos: {n_nodos}  |  Mejor valor: {mejor_val}',
        fontsize=11, color=TEXTO)

anim = FuncAnimation(fig_a, _update_anim, frames=len(pasos_anim),
                     interval=600, repeat=False)
plt.close(fig_a)

try:
    display(HTML(anim.to_jshtml()))
except Exception as e:
    print(f'Animación no disponible en este entorno: {e}')
    _update_anim(len(pasos_anim) - 1)
    plt.show()

In [ ]:
# ── Problema de Asignación con Branch & Bound ──────────────────────────────
#
# Dado: n trabajadores, n tareas, matriz de costos cost[i][j].
# Objetivo: asignar cada trabajador a una tarea distinta minimizando el costo total.
# Cota inferior: reducción de filas y columnas (Hungarian method parcial).

def reducir_matriz(mat: List[List[int]]) -> Tuple[List[List[int]], int]:
    '''
    Reduce una matriz de costos: resta el mínimo de cada fila y columna.
    Retorna (matriz_reducida, cota_inferior).
    '''
    n = len(mat)
    m = [row[:] for row in mat]
    lb = 0
    for i in range(n):
        mn = min(m[i])
        lb += mn
        m[i] = [v - mn for v in m[i]]
    for j in range(n):
        mn = min(m[i][j] for i in range(n))
        lb += mn
        for i in range(n):
            m[i][j] -= mn
    return m, lb


def asignacion_bb(
        costo: List[List[int]],
        nombres_trab: Optional[List[str]] = None,
        nombres_tarea: Optional[List[str]] = None,
        verbose: bool = True
) -> Tuple[int, List[int]]:
    '''
    Problema de Asignación resuelto con Branch & Bound.

    Estrategia: DFS + cota inferior por reducción de sub-matriz.
    En cada nodo se asigna el trabajador `nivel` a alguna tarea libre.
    La cota es el costo acumulado + LB de la sub-matriz de lo que queda.

    Complejidad:
        Tiempo: O(n!) peor caso, en práctica mucho mejor por las cotas
        Espacio: O(n) recursión + O(nodos) árbol
    '''
    n = len(costo)
    if nombres_trab is None:
        nombres_trab  = [f'W{i+1}' for i in range(n)]
    if nombres_tarea is None:
        nombres_tarea = [f'T{j+1}' for j in range(n)]

    mejor_costo = [sum(costo[i][i] for i in range(n))]  # diagonal como cota inicial
    mejor_asig: List[List[int]] = [[list(range(n))[i] for i in range(n)]]
    nodos_exp = [0]
    nodos_pod = [0]

    def dfs(nivel: int, asig: List[int], disponibles: List[int],
            costo_acum: int) -> None:
        nodos_exp[0] += 1
        if nivel == n:
            if costo_acum < mejor_costo[0]:
                mejor_costo[0] = costo_acum
                mejor_asig[0] = asig[:]
            return
        for tarea in disponibles:
            nuevo_costo = costo_acum + costo[nivel][tarea]
            nuevas_disp = [t for t in disponibles if t != tarea]
            # Calcular cota inferior para el resto
            if nuevas_disp:
                filas_rest = list(range(nivel + 1, n))
                sub = [[costo[i][j] for j in nuevas_disp] for i in filas_rest]
                _, lb_sub = reducir_matriz(sub)
            else:
                lb_sub = 0
            lb_total = nuevo_costo + lb_sub
            if lb_total < mejor_costo[0]:
                dfs(nivel + 1, asig + [tarea], nuevas_disp, nuevo_costo)
            else:
                nodos_pod[0] += 1

    dfs(0, [], list(range(n)), 0)

    if verbose:
        print('Problema de Asignación — Branch & Bound')
        print(f'  Costo óptimo : {mejor_costo[0]}')
        asig_str = ', '.join(
            f'{nombres_trab[i]}→{nombres_tarea[mejor_asig[0][i]]}'
            for i in range(n)
        )
        print(f'  Asignación   : {asig_str}')
        print(f'  Nodos explor.: {nodos_exp[0]}')
        print(f'  Nodos podados: {nodos_pod[0]}')

    return mejor_costo[0], mejor_asig[0]


# Ejemplo — matriz 4×4 (clásico de libros de texto)
COSTO_ASIG = [
    [9, 2, 7, 8],
    [6, 4, 3, 7],
    [5, 8, 1, 8],
    [7, 6, 9, 4],
]
costo_opt, asig_opt = asignacion_bb(
    COSTO_ASIG,
    nombres_trab=['Ana', 'Bea', 'Carlos', 'Diana'],
    nombres_tarea=['Tarea1', 'Tarea2', 'Tarea3', 'Tarea4']
)

In [ ]:
# ── Visualización del Problema de Asignación ────────────────────────────────
nombres_trab  = ['Ana', 'Bea', 'Carlos', 'Diana']
nombres_tarea = ['Tarea1', 'Tarea2', 'Tarea3', 'Tarea4']
n_asig = len(COSTO_ASIG)

fig_asig, (ax_mat, ax_asig) = plt.subplots(1, 2, figsize=(13, 5))
fig_asig.patch.set_facecolor(FONDO)
fig_asig.suptitle('Problema de Asignación — Solución Branch & Bound',
                   fontsize=13, fontweight='bold', color=TEXTO)

# Panel 1: Matriz de costos con solución resaltada
cmap_data = np.array(COSTO_ASIG, dtype=float)
im = ax_mat.imshow(cmap_data, cmap='Blues', aspect='auto', alpha=0.6)
for i in range(n_asig):
    for j in range(n_asig):
        v = COSTO_ASIG[i][j]
        es_optimo = (asig_opt[i] == j)
        fc = VERDE if es_optimo else 'white'
        ax_mat.add_patch(plt.Rectangle((j - 0.5, i - 0.5), 1, 1,
                                        facecolor=fc, alpha=0.5, zorder=1))
        fw = 'bold' if es_optimo else 'normal'
        ax_mat.text(j, i, str(v), ha='center', va='center',
                    fontsize=13, color=TEXTO, fontweight=fw, zorder=2)

ax_mat.set_xticks(range(n_asig))
ax_mat.set_yticks(range(n_asig))
ax_mat.set_xticklabels(nombres_tarea, fontsize=9, color=TEXTO)
ax_mat.set_yticklabels(nombres_trab, fontsize=9, color=TEXTO)
ax_mat.set_title('Matriz de costos (verde = asignación óptima)', color=TEXTO, fontsize=10)
ax_mat.set_facecolor(FONDO)

# Panel 2: Bipartito — trabajadores a tareas
ax_asig.set_facecolor(FONDO)
ax_asig.axis('off')
for i, trab in enumerate(nombres_trab):
    yi = 1 - i / (n_asig - 1)
    ax_asig.text(0.05, yi, trab, ha='left', va='center',
                 fontsize=12, color=TEXTO,
                 bbox=dict(boxstyle='round,pad=0.4', facecolor=AZ_CLARO, alpha=0.8))

for j, tarea in enumerate(nombres_tarea):
    yj = 1 - j / (n_asig - 1)
    ax_asig.text(0.95, yj, tarea, ha='right', va='center',
                 fontsize=12, color=TEXTO,
                 bbox=dict(boxstyle='round,pad=0.4', facecolor=AZ_CLARO, alpha=0.8))

for i in range(n_asig):
    yi = 1 - i / (n_asig - 1)
    j = asig_opt[i]
    yj = 1 - j / (n_asig - 1)
    costo_ij = COSTO_ASIG[i][j]
    ax_asig.annotate(
        '', xy=(0.88, yj), xytext=(0.12, yi),
        arrowprops=dict(arrowstyle='->', color=VERDE, lw=2.5))
    ax_asig.text(0.5, (yi + yj) / 2 + 0.04 * (1 if i % 2 == 0 else -1),
                 f'costo={costo_ij}', ha='center', va='center',
                 fontsize=8, color=MORADO)

ax_asig.text(0.5, -0.08, f'Costo total óptimo: {costo_opt}',
             ha='center', va='center', fontsize=12, fontweight='bold',
             color=TEXTO, transform=ax_asig.transAxes,
             bbox=dict(boxstyle='round,pad=0.4', facecolor=VERDE, alpha=0.3))
ax_asig.set_title('Asignación óptima encontrada', color=TEXTO, fontsize=10)
ax_asig.set_xlim(0, 1)
ax_asig.set_ylim(-0.2, 1.15)

plt.tight_layout()
plt.show()

## 4. Análisis de complejidad

### Branch & Bound vs otros paradigmas para mochila 0/1

| Algoritmo | Tiempo (peor caso) | Tiempo (caso promedio) | Óptimo | Espacio |
|---|---|---|---|---|
| Greedy (NB02) | O(n log n) | O(n log n) | ✗ (no siempre) | O(1) |
| DP (NB03) | O(n·W) | O(n·W) | ✓ | O(n·W) |
| Backtracking (NB04) | O(2ⁿ) | O(2ⁿ) | ✓ | O(n) |
| Branch & Bound (NB05) | O(2ⁿ) | **mucho < 2ⁿ** | ✓ | O(n) |

### ¿Cuándo conviene B&B sobre DP?

- **DP** es preferible cuando W es pequeño o moderado (pseudo-polinomial O(n·W))
- **B&B** es preferible cuando W es muy grande (donde la tabla DP sería enorme)
  - Ej: n=30 objetos, W=10⁹ → DP requiere 30 × 10⁹ celdas = inviable
  - B&B puede resolver este caso si la función de cota es buena

### Calidad de la cota y eficiencia

La eficiencia de B&B depende criticamente de cuán **ajustada** es la función de cota:

- **Cota perfecta** (= óptimo real) → solo se explora el camino óptimo: O(n)
- **Cota pésima** (= ∞ siempre) → igual que Backtracking puro: O(2ⁿ)
- **Cota fraccionaria** (como la nuestra) → en práctica explora una fracción pequeña del árbol

In [ ]:
# ── Gráfico: nodos explorados por tamaño n ─────────────────────────────────
import random
random.seed(42)

def generar_objetos(n: int, seed: int = 42) -> List[Dict]:
    rng = random.Random(seed)
    return [{'nombre': f'O{i+1}',
             'valor': rng.randint(10, 80),
             'peso':  rng.randint(10, 60)} for i in range(n)]

ns = list(range(1, 16))
nodos_bt_list = []
nodos_bb_list = []

for n_i in ns:
    objs_i = generar_objetos(n_i)
    cap_i  = sum(o['peso'] for o in objs_i) // 2
    _, _, tree_bt = mochila_bt_con_arbol(objs_i, cap_i)
    _, _, tree_bb = mochila_bb_con_arbol(objs_i, cap_i)
    nodos_bt_list.append(len(tree_bt))
    nodos_bb_list.append(len(tree_bb))

fig_comp, ax_c = plt.subplots(figsize=(10, 5))
fig_comp.patch.set_facecolor(FONDO)
ax_c.set_facecolor(FONDO)

ax_c.plot(ns, nodos_bt_list, 'o-', color=NARANJA, lw=2, ms=6, label='Backtracking')
ax_c.plot(ns, nodos_bb_list, 's-', color=AZUL,    lw=2, ms=6, label='Branch & Bound')
ax_c.fill_between(ns, nodos_bb_list, nodos_bt_list,
                   alpha=0.15, color=VERDE, label='Nodos ahorrados')

ax_c.set_xlabel('Número de objetos (n)', color=TEXTO, fontsize=11)
ax_c.set_ylabel('Nodos explorados', color=TEXTO, fontsize=11)
ax_c.set_title('Nodos explorados: Backtracking vs Branch & Bound\n'
               '(objetos generados aleatoriamente, capacidad = suma_pesos / 2)',
               color=TEXTO, fontsize=11)
ax_c.legend(fontsize=10)
ax_c.set_yscale('log')
ax_c.tick_params(colors=TEXTO)
ax_c.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── Widget: Explorador B&B vs Backtracking ─────────────────────────────────

def mochila_configurable(
        objetos: List[Dict], capacidad: int, usar_cota: bool = True
) -> Tuple[int, List[str], int, int]:
    '''Mochila 0/1 configurable: con o sin función de cota.'''
    objs = ordenar_por_ratio(objetos)
    n = len(objs)
    mejor = [0]
    mejor_items: List[List[str]] = [[]]
    explorados = [0]
    podados    = [0]

    def explorar(nivel, peso, valor, items):
        explorados[0] += 1
        if valor > mejor[0]:
            mejor[0] = valor
            mejor_items[0] = items[:]
        if nivel >= n:
            return
        obj = objs[nivel]
        nv = nivel + 1
        np_ = peso + obj['peso']
        nv_ = valor + obj['valor']

        # Rama incluir
        if np_ <= capacidad:
            if usar_cota:
                c = cota_superior_fraccionaria(nv, np_, nv_, objs, capacidad)
                if c > mejor[0]:
                    explorar(nv, np_, nv_, items + [obj['nombre']])
                else:
                    podados[0] += 1
            else:
                explorar(nv, np_, nv_, items + [obj['nombre']])
        else:
            podados[0] += 1

        # Rama excluir
        if usar_cota:
            c = cota_superior_fraccionaria(nv, peso, valor, objs, capacidad)
            if c > mejor[0]:
                explorar(nv, peso, valor, items)
            else:
                podados[0] += 1
        else:
            explorar(nv, peso, valor, items)

    explorar(0, 0, 0, [])
    return mejor[0], mejor_items[0], explorados[0], podados[0]


n_sl = IntSlider(value=5, min=1, max=5, step=1,
                 description='Objetos:',
                 style={'description_width': '80px'})
W_sl = IntSlider(value=300, min=50, max=450, step=25,
                 description='Capacidad W:',
                 style={'description_width': '100px'})
cota_cb = Checkbox(value=True, description='Usar cota superior (B&B)',
                   indent=False)
out_w = Output()

def _actualizar_widget(change=None):
    with out_w:
        out_w.clear_output(wait=True)
        objs = OBJETOS[:n_sl.value]
        W    = W_sl.value
        usar_cota = cota_cb.value

        v_bt, _, nb_bt, np_bt = mochila_configurable(objs, W, usar_cota=False)
        v_alg, items_alg, nb_alg, np_alg = mochila_configurable(objs, W, usar_cota=usar_cota)

        label_alg = 'Branch & Bound' if usar_cota else 'Backtracking'

        fig_w, (axA, axB) = plt.subplots(1, 2, figsize=(12, 4))
        fig_w.patch.set_facecolor(FONDO)

        # Gráfico de barras: nodos explorados
        cats  = ['Backtracking\npuro', label_alg]
        vals  = [nb_bt, nb_alg]
        cols  = [NARANJA, AZUL if usar_cota else NARANJA]
        bars  = axA.bar(cats, vals, color=cols, edgecolor='white', width=0.5)
        for bar, v in zip(bars, vals):
            axA.text(bar.get_x() + bar.get_width() / 2,
                     bar.get_height() + 0.3,
                     str(v), ha='center', va='bottom',
                     fontsize=12, fontweight='bold', color=TEXTO)
        if usar_cota and nb_bt > 0:
            red = (1 - nb_alg / nb_bt) * 100
            axA.text(0.5, 0.95, f'Reducción: {red:.1f}%',
                     transform=axA.transAxes, ha='center', va='top',
                     fontsize=11, color=VERDE,
                     bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.85))
        axA.set_facecolor(FONDO)
        axA.set_ylabel('Nodos explorados', color=TEXTO)
        axA.set_title(f'{n_sl.value} objetos, W={W}', fontsize=11, color=TEXTO)
        axA.tick_params(colors=TEXTO)

        # Panel de detalles
        axB.axis('off')
        axB.set_facecolor(FONDO)
        items_str = ', '.join(items_alg) if items_alg else '(ninguno)'
        info = (
            f'Algoritmo: {label_alg}\n\n'
            f'Valor óptimo : {v_alg}\n'
            f'Objetos      : {items_str}\n\n'
            f'Nodos explor.: {nb_alg}\n'
            f'Nodos podados: {np_alg}'
        )
        axB.text(0.05, 0.92, info, transform=axB.transAxes,
                 fontsize=10, va='top', family='monospace', color=TEXTO,
                 bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.9))
        axB.set_title('Resultado', fontsize=11, color=TEXTO)

        plt.tight_layout()
        plt.show()


n_sl.observe(_actualizar_widget, names='value')
W_sl.observe(_actualizar_widget, names='value')
cota_cb.observe(_actualizar_widget, names='value')

display(VBox([
    widgets.HTML(value='<h3 style="color:#212121;margin-bottom:6px">'
                       'Explorador: Backtracking vs Branch &amp; Bound</h3>'),
    HBox([n_sl, W_sl]),
    cota_cb,
    out_w
]))
_actualizar_widget()

---
## 5. Ejercicios

### Ejercicio ⭐ — Mochila B&B desde cero

**Enunciado:** Implementa la función `mochila_bb(pesos, valores, W)` que resuelva la mochila 0/1 usando Branch & Bound.
Usa los mismos objetos del PDF (con los mismos datos usados en NB02, NB03 y NB04) para cerrar el ciclo comparativo de los 4 paradigmas.

- Input: `pesos: List[int]`, `valores: List[int]`, `W: int`
- Output: `(valor_maximo: int, items_incluidos: List[int])` donde `items_incluidos` es la lista de índices 0-based

Muestra la tabla comparativa final con los 4 paradigmas.

**Pista:** Ordena objetos por ratio valor/peso antes de calcular la cota superior. La cota fraccionaria es: toma todos los objetos que quepan enteros y una fracción del último.

In [ ]:
# Datos del PDF para las 4 comparaciones
PESOS_PDF   = [50, 100, 150, 200, 250]
VALORES_PDF = [20,  24,  55,  40,  70]
W_PDF       = 300

def mochila_bb(pesos: List[int], valores: List[int], W: int):
    '''
    TODO: implementa Branch & Bound para la mochila 0/1.

    Pasos sugeridos:
    1. Ordena objetos por ratio valor/peso descendente.
    2. Define cota_sup(nivel, peso_acum, valor_acum) usando relajación fraccionaria.
    3. Implementa DFS recursivo: en cada nodo calcula cota_sup antes de explorar.
    4. Poda si cota_sup <= mejor_valor_hasta_ahora.
    '''
    n = len(pesos)
    # Tu código aquí
    raise NotImplementedError('Implementa mochila_bb')


# Prueba rápida:
# val, items = mochila_bb(PESOS_PDF, VALORES_PDF, W_PDF)
# print(f'Valor: {val}, Índices: {items}')

In [ ]:
# ── Verificador Ejercicio ⭐ ─────────────────────────────────────────────────
def _verificar_ej1():
    casos = [
        ([50, 100, 150, 200, 250], [20, 24, 55, 40, 70], 300, 79),  # datos PDF
        ([10, 20, 30],             [60, 100, 120],        50,  220),
        ([1, 2, 3],                [1,  2,   3],          4,   5),
        ([5],                      [10],                  5,   10),
        ([5],                      [10],                  4,   0),
    ]
    ok = True
    for pesos, valores, W, esperado in casos:
        try:
            val, items = mochila_bb(pesos, valores, W)
        except NotImplementedError:
            print('✗ NotImplementedError — implementa la función')
            return
        except Exception as e:
            print(f'✗ Error inesperado: {e}')
            return
        if val != esperado:
            print(f'✗ pesos={pesos}, W={W}: esperado {esperado}, obtenido {val}')
            ok = False
            continue
        # Verificar que los índices son válidos y consistentes
        peso_total = sum(pesos[i] for i in items)
        val_total  = sum(valores[i] for i in items)
        if peso_total > W:
            print(f'✗ Solución viola la capacidad: peso={peso_total} > W={W}')
            ok = False
        elif val_total != val:
            print(f'✗ Valor declarado ({val}) no coincide con suma de items ({val_total})')
            ok = False
        else:
            print(f'✓ pesos={pesos}, W={W} → valor={val}')
    if ok:
        print('\n✅ ¡Todos los casos correctos!')
        print('Ciclo mochila completo: Greedy(NB02) → DP(NB03) → BT(NB04) → B&B(NB05)')

_verificar_ej1()

In [ ]:
# ── Solución Ejercicio ⭐ (descomentar para ver) ────────────────────────────
# def mochila_bb(pesos, valores, W):
#     n = len(pesos)
#     orden = sorted(range(n), key=lambda i: valores[i]/pesos[i], reverse=True)
#     p_ord = [pesos[i]   for i in orden]
#     v_ord = [valores[i] for i in orden]
#
#     mejor = [0]
#     mejor_idx = [[]]
#
#     def cota(niv, peso_ac, val_ac):
#         if peso_ac > W:
#             return 0.0
#         b = float(val_ac)
#         r = W - peso_ac
#         for k in range(niv, n):
#             if p_ord[k] <= r:
#                 r -= p_ord[k]
#                 b += v_ord[k]
#             else:
#                 b += (r / p_ord[k]) * v_ord[k]
#                 break
#         return b
#
#     def bb(niv, peso, val, idx_ord):
#         if val > mejor[0]:
#             mejor[0] = val
#             mejor_idx[0] = [orden[k] for k in idx_ord]
#         if niv >= n:
#             return
#         # Incluir
#         np_ = peso + p_ord[niv]
#         nv_ = val  + v_ord[niv]
#         if np_ <= W:
#             if cota(niv+1, np_, nv_) > mejor[0]:
#                 bb(niv+1, np_, nv_, idx_ord+[niv])
#         # Excluir
#         if cota(niv+1, peso, val) > mejor[0]:
#             bb(niv+1, peso, val, idx_ord)
#
#     bb(0, 0, 0, [])
#     return mejor[0], sorted(mejor_idx[0])
print('Solución comentada — descomentar para ver.')

---
### Ejercicio ⭐⭐ — 8-Puzzle con Branch & Bound y distancia Manhattan

**Enunciado:** Dado un tablero 3×3 con los números 1–8 y un espacio vacío (0),
encuentra la secuencia de movimientos que lleva al estado objetivo.

```
Estado inicial ejemplo:    Estado objetivo:
  1  2  5                   1  2  3
  3  4  0    →→→→→→→        4  5  6
  6  7  8                   7  8  0
```

- **Heurística de cota**: distancia Manhattan de cada pieza a su posición objetivo
  (suma de |fila_actual - fila_obj| + |col_actual - col_obj| para cada pieza)
- **Estrategia**: best-first search con f = g + h (g = pasos dados, h = Manhattan)
- **Input**: `estado_inicial: List[int]` (9 enteros, 0 = espacio vacío)
- **Output**: `(movimientos: List[str], nodos_explorados: int)` o `(None, n)` si no tiene solución

**Pista:** Usa `heapq` para la cola de prioridad. Para evitar ciclos, guarda los estados visitados en un `dict` con el costo mínimo `g` con que se llegó a cada uno.

In [ ]:
def distancia_manhattan(estado: tuple, objetivo: tuple = None) -> int:
    '''
    Suma de distancias Manhattan de cada pieza a su posición objetivo.
    La pieza 0 (espacio vacío) no se cuenta.
    '''
    if objetivo is None:
        objetivo = tuple(range(1, 9)) + (0,)  # (1,2,3,4,5,6,7,8,0)
    total = 0
    for i, val in enumerate(estado):
        if val == 0:
            continue
        # Tu código aquí: calcula distancia Manhattan de la pieza 'val'
        pass
    return total


def resolver_8puzzle(estado_inicial: List[int]):
    '''
    Resuelve el 8-puzzle con Branch & Bound (A* / best-first search).

    Pasos:
    1. Define objetivo = (1,2,3,4,5,6,7,8,0).
    2. Cola de prioridad: (f, g, estado, camino_movs).
    3. Expande nodo con menor f = g + h (h = distancia Manhattan).
    4. Para evitar ciclos: diccionario visitados con g mínimo.
    5. Posibles movimientos del espacio vacío: arriba, abajo, izquierda, derecha.
    '''
    objetivo = tuple(range(1, 9)) + (0,)
    inicio   = tuple(estado_inicial)

    if inicio == objetivo:
        return [], 0

    # Tu código aquí
    raise NotImplementedError('Implementa resolver_8puzzle')


# Prueba rápida:
# movs, nodos = resolver_8puzzle([1, 2, 5, 3, 4, 0, 6, 7, 8])
# print(f'{nodos} nodos explorados, solución: {movs}')

In [ ]:
# ── Verificador Ejercicio ⭐⭐ ────────────────────────────────────────────────
def _aplicar_movimiento(estado: tuple, mov: str) -> Optional[tuple]:
    lst = list(estado)
    pos = lst.index(0)
    fila, col = pos // 3, pos % 3
    deltas = {'arriba': -3, 'abajo': 3, 'izquierda': -1, 'derecha': 1}
    validos = {'arriba': fila > 0, 'abajo': fila < 2,
               'izquierda': col > 0, 'derecha': col < 2}
    if mov not in deltas or not validos[mov]:
        return None
    nueva = pos + deltas[mov]
    lst[pos], lst[nueva] = lst[nueva], lst[pos]
    return tuple(lst)


def _verificar_ej2():
    objetivo = tuple(range(1, 9)) + (0,)
    casos = [
        [1, 2, 5, 3, 4, 0, 6, 7, 8],   # 3 pasos
        [1, 2, 3, 4, 5, 6, 7, 8, 0],   # ya resuelto (0 pasos)
        [1, 2, 3, 4, 5, 6, 0, 7, 8],   # 2 pasos
    ]
    ok = True
    for caso in casos:
        try:
            movs, nodos = resolver_8puzzle(caso)
        except NotImplementedError:
            print('✗ NotImplementedError — implementa la función')
            return
        except Exception as e:
            print(f'✗ Error inesperado: {e}')
            return

        # Replay
        estado = tuple(caso)
        for m in (movs or []):
            nuevo = _aplicar_movimiento(estado, m)
            if nuevo is None:
                print(f'✗ Movimiento inválido "{m}" desde {estado}')
                ok = False
                break
            estado = nuevo
        else:
            if estado != objetivo:
                print(f'✗ No llega al objetivo. Estado final: {estado}')
                ok = False
            else:
                pasos = len(movs) if movs else 0
                print(f'✓ {caso} → {pasos} movs, {nodos} nodos explorados')

    if ok:
        print('\n✅ ¡Todos los casos correctos!')

_verificar_ej2()

In [ ]:
# ── Solución Ejercicio ⭐⭐ (descomentar para ver) ───────────────────────────
# def distancia_manhattan(estado, objetivo=None):
#     if objetivo is None:
#         objetivo = tuple(range(1, 9)) + (0,)
#     total = 0
#     for i, val in enumerate(estado):
#         if val == 0:
#             continue
#         pos_obj = objetivo.index(val)
#         total += abs(i // 3 - pos_obj // 3) + abs(i % 3 - pos_obj % 3)
#     return total
#
# def resolver_8puzzle(estado_inicial):
#     objetivo = tuple(range(1, 9)) + (0,)
#     inicio   = tuple(estado_inicial)
#     if inicio == objetivo:
#         return [], 0
#     h0  = distancia_manhattan(inicio)
#     cola = [(h0, 0, inicio, [])]
#     visitados = {}
#     nodos_exp = 0
#     deltas = [(-3, 'arriba'), (3, 'abajo'), (-1, 'izquierda'), (1, 'derecha')]
#     while cola:
#         f, g, estado, camino = heapq.heappop(cola)
#         if estado in visitados and visitados[estado] <= g:
#             continue
#         visitados[estado] = g
#         nodos_exp += 1
#         if estado == objetivo:
#             return camino, nodos_exp
#         pos  = estado.index(0)
#         fila, col = pos // 3, pos % 3
#         mov_validos = []
#         if fila > 0: mov_validos.append((-3, 'arriba'))
#         if fila < 2: mov_validos.append((3,  'abajo'))
#         if col  > 0: mov_validos.append((-1, 'izquierda'))
#         if col  < 2: mov_validos.append((1,  'derecha'))
#         for delta, nombre in mov_validos:
#             lst = list(estado)
#             lst[pos], lst[pos + delta] = lst[pos + delta], lst[pos]
#             nuevo = tuple(lst)
#             ng = g + 1
#             if nuevo not in visitados or visitados[nuevo] > ng:
#                 heapq.heappush(cola, (ng + distancia_manhattan(nuevo), ng, nuevo, camino + [nombre]))
#     return None, nodos_exp
print('Solución comentada — descomentar para ver.')

---
## 6. Práctica libre

Usa las celdas siguientes para experimentar:

1. **Más objetos, misma capacidad:** Aumenta OBJETOS a 10-15 items y compara nodos BT vs B&B. ¿Qué tan rápido crece la diferencia?

2. **Cota perezosa vs cota ajustada:** Implementa una cota que simplemente retorne `valor + sum_valores_restantes` (ignorando el peso). Es admisible pero muy imprecisa. ¿Cuántos más nodos explora vs la cota fraccionaria?

3. **8-puzzle difícil:** Prueba con `[8, 7, 6, 5, 4, 3, 2, 1, 0]`. ¿Cuántos nodos explora? ¿Tarda mucho? (Atención: algunos estados del 8-puzzle son irresolubles — el estado anterior tiene solución.)

4. **Problema de asignación 5×5:** Genera una matriz aleatoria 5×5 y resuelve con `asignacion_bb`. Compara el resultado con fuerza bruta (`itertools.permutations`).

In [ ]:
# Espacio para práctica libre


In [ ]:
# ── Autoevaluación ──────────────────────────────────────────────────────────
import ipywidgets as wg

PREGUNTAS = [
    {
        'enunciado': '1. ¿Cuál es la diferencia fundamental entre Backtracking y Branch & Bound?',
        'opciones': [
            'B&B usa BFS; Backtracking usa DFS',
            'B&B poda ramas subóptimas además de las inviables',
            'B&B solo funciona para problemas de grafos',
            'Backtracking garantiza el óptimo; B&B no',
        ],
        'correcta': 1,
        'exp': 'B&B agrega poda por suboptimalidad (ramas que no pueden superar el mejor valor actual), además de la poda por inviabilidad que ya hace Backtracking.',
    },
    {
        'enunciado': '2. La cota superior fraccionaria para la mochila es ADMISIBLE porque:',
        'opciones': [
            'Siempre subestima el valor real',
            'Es igual al valor óptimo real',
            'Nunca subestima el valor real (la versión fraccionaria >= 0/1)',
            'Es O(1) de calcular',
        ],
        'correcta': 2,
        'exp': 'La versión fraccionaria es más permisiva que la 0/1 (puede tomar fracciones), por lo que su óptimo siempre es >= al óptimo 0/1. Esto garantiza que nunca descartamos la solución óptima real.',
    },
    {
        'enunciado': '3. Para la mochila 0/1 con n=30 objetos y W=10^9, ¿qué paradigma es más eficiente?',
        'opciones': [
            'Greedy (O(n log n))',
            'DP tabla 2D (O(n·W))',
            'Backtracking (O(2^n))',
            'Branch & Bound (explora fracción del árbol)',
        ],
        'correcta': 3,
        'exp': 'Con W=10^9, la tabla DP requeriría 30×10^9 celdas ≈ 30 GB de memoria. B&B explora solo una fracción del árbol de 2^30 posibilidades, lo que en la práctica es mucho más manejable.',
    },
    {
        'enunciado': '4. En el problema del 8-puzzle, la heurística de distancia Manhattan es ADMISIBLE porque:',
        'opciones': [
            'Nunca subestima los movimientos necesarios',
            'Nunca sobreestima los movimientos necesarios',
            'Calcula exactamente los movimientos necesarios',
            'Es siempre mayor que el número real de pasos',
        ],
        'correcta': 1,
        'exp': 'La distancia Manhattan cuenta cuántos pasos mínimos necesitaría cada pieza si pudiera moverse independientemente. Como en la práctica las piezas se interfieren entre sí, el número real de pasos es siempre >= Manhattan. Nunca sobreestima → admisible.',
    },
    {
        'enunciado': '5. ¿Qué ocurre si la función de cota siempre retorna +infinito?',
        'opciones': [
            'B&B encuentra la solución en O(1)',
            'B&B nunca encuentra solución',
            'B&B se comporta exactamente como Backtracking puro',
            'B&B explora solo la primera rama',
        ],
        'correcta': 2,
        'exp': 'Si la cota siempre es mayor que cualquier solución actual, la condición de poda (cota <= mejor_actual) nunca se cumple. Resultado: se exploran todos los nodos del árbol, igual que Backtracking sin poda por suboptimalidad.',
    },
]

_selecciones = {}
_salida_quiz = Output()

controles = []
for qi, q in enumerate(PREGUNTAS):
    rb = wg.RadioButtons(
        options=q['opciones'],
        description='',
        layout=wg.Layout(width='100%'),
    )
    _selecciones[qi] = rb
    controles.append(wg.HTML(value=f'<b>{q["enunciado"]}</b>'))
    controles.append(rb)
    controles.append(wg.HTML(value='<hr style="margin:4px 0"/>'))

btn_quiz = wg.Button(description='Evaluar respuestas',
                     button_style='primary', icon='check')

def _evaluar_quiz(b):
    with _salida_quiz:
        _salida_quiz.clear_output()
        score = 0
        for qi, q in enumerate(PREGUNTAS):
            rb = _selecciones[qi]
            idx = q['opciones'].index(rb.value)
            if idx == q['correcta']:
                score += 1
                print(f'✅ {qi+1}. Correcto!')
            else:
                corr = q['opciones'][q['correcta']]
                print(f'❌ {qi+1}. Incorrecto. Respuesta: "{corr}"')
            print(f'   → {q["exp"]}')
        pct = score / len(PREGUNTAS) * 100
        print(f'\nPuntaje: {score}/{len(PREGUNTAS)} ({pct:.0f}%)')
        if pct == 100:
            print('¡Excelente! Dominas Branch & Bound.')
        elif pct >= 60:
            print('Bien. Repasa los conceptos de cota admisible y comparación con DP.')
        else:
            print('Repasa la sección de teoría y las comparaciones del widget.')

btn_quiz.on_click(_evaluar_quiz)
display(VBox(controles + [btn_quiz, _salida_quiz]))

---
## 7. Lecturas recomendadas

### Libros de texto

| Recurso | Capítulo | Tema |
|---|---|---|
| **Skiena** — *Algorithm Design Manual* (3ª ed.) | Cap. 9 — *Combinatorial Search* | Branch & Bound extenso, ejemplos con TSP y mochila |
| **CLRS** — *Introduction to Algorithms* (4ª ed.) | No tiene capítulo específico | Ver Cap. 14 (DP) + Apéndice para trasfondo |
| **Kleinberg & Tardos** — *Algorithm Design* (1ª ed.) | Cap. 7 parcial | Network Flow con búsqueda exhaustiva |

### Problemas de práctica

Branch & Bound puro tiene pocos problemas específicos en jueces online. Los problemas que lo trabajan suelen aparecer con etiquetas mixtas:

| Plataforma | Búsqueda sugerida | Nivel |
|---|---|---|
| Codeforces | Tag `brute force` + rating 1200–1400 | Medio |
| Codeforces | Tag `dp` + rating 1400–1600 (problemas donde B&B compite con DP) | Desafío |
| AtCoder Educational DP Contest | Problemas D (Knapsack 1), E (Knapsack 2) | Directo |
| USACO Guide | Sección *Complete Search* + *Optimizations* | Silver/Gold |

### Recursos online

- **VisuAlgo** — [visualgo.net/en](https://visualgo.net/en): visualizaciones interactivas de algoritmos
- **CP-Algorithms** — [cp-algorithms.com](https://cp-algorithms.com): explicaciones con código, incluyendo B&B
- **USACO Guide** — [usaco.guide](https://usaco.guide): currículo estructurado, sección *Complete Search with Pruning*

---
### Resumen: los 4 paradigmas para la mochila 0/1

| Paradigma | NB | Tiempo | Óptimo | Nodos (5 objetos, W=300) |
|---|---|---|---|---|
| Greedy | 02 | O(n log n) | ✗ no siempre | n/a |
| DP | 03 | O(n·W) | ✓ | n/a |
| Backtracking | 04 | O(2ⁿ) | ✓ | ~63 |
| **Branch & Bound** | **05** | **O(2ⁿ) / mucho menos** | **✓** | **~20** |

> **Próximo:** NB06 — *Más allá del curso*: Algoritmos de Aproximación, Metaheurísticas y Algoritmos Aleatorizados.